In [1]:
import pandas as pd

In [2]:
caminho_produto = 'C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/processed/produtos_clean.csv'
df_produto = pd.read_csv(caminho_produto)

caminho_vendas = 'C:/Users/hiury/OneDrive/Área de Trabalho/IFCE/projeto_indicium/datasets/processed/vendas_clean.csv'
df_vendas = pd.read_csv(caminho_vendas)

In [6]:
id_produto = df_produto[
    df_produto['name'] == 'Motor de Popa Yamaha Evo Dash 155HP'
]['id_product'].values[0]

print(id_produto)

54


In [9]:
df_produto_vendas = df_vendas[df_vendas['id_product'] == id_produto].copy()
df_produto_vendas['sale_date'] = pd.to_datetime(df_produto_vendas['sale_date'], format='mixed')

df_produto_vendas

,id,id_client,id_product,qtd,total,sale_date
46,48,13,54,15,1823022.00,2024-05-30
53,55,35,54,3,346373.80,2024-11-24
71,74,45,54,11,1270038.85,2024-09-25
442,451,42,54,13,1500955.35,2024-02-19
494,503,45,54,11,1270038.85,2024-11-27
...,...,...,...,...,...,...
9210,9308,20,54,2,230916.50,2024-06-25
9365,9464,20,54,12,1385497.10,2024-05-27
9426,9526,37,54,15,1731870.90,2023-02-27
9508,9609,19,54,13,1500955.35,2023-05-16


In [10]:
df_diario = df_produto_vendas.groupby(
    df_produto_vendas['sale_date'].dt.date
)['qtd'].sum().reset_index()
df_diario.columns = ['data', 'qtd']
df_diario['data'] = pd.to_datetime(df_diario['data'])

# Calendário completo do período
calendario = pd.date_range(start='2023-01-01', end='2024-12-31', freq='D')
df_completo = pd.DataFrame({'data': calendario})
df_completo = df_completo.merge(df_diario, on='data', how='left').fillna(0)

df_treino = df_completo[df_completo['data'] <= '2023-12-31'].copy()
df_teste  = df_completo[
    (df_completo['data'] >= '2024-01-01') &
    (df_completo['data'] <= '2024-01-31')
].copy()

print(f'Dias no treino: {len(df_treino)}')
print(f'Dias no teste:  {len(df_teste)}')

Dias no treino: 365
Dias no teste:  31


In [ ]:
previsoes = []
for data in df_teste['data']:
    janela = df_completo[df_completo['data'] < data].tail(7)['qtd']
    previsoes.append(round(janela.mean(), 2))

df_teste['previsao'] = previsoes
df_teste['erro_abs'] = abs(df_teste['qtd'] - df_teste['previsao'])

In [12]:
mae = round(df_teste['erro_abs'].mean(), 2)
print(f'\nMAE: {mae} unidades')
print(f'\nPrevisões vs Real:')
print(df_teste[['data', 'qtd', 'previsao', 'erro_abs']].to_string(index=False))


MAE: 1.64 unidades

Previsões vs Real:
      data  qtd  previsao  erro_abs
2024-01-01  0.0      0.00      0.00
2024-01-02  0.0      0.00      0.00
2024-01-03  0.0      0.00      0.00
2024-01-04  0.0      0.00      0.00
2024-01-05  0.0      0.00      0.00
2024-01-06  0.0      0.00      0.00
2024-01-07  0.0      0.00      0.00
2024-01-08  0.0      0.00      0.00
2024-01-09  0.0      0.00      0.00
2024-01-10 10.0      0.00     10.00
2024-01-11  0.0      1.43      1.43
2024-01-12  0.0      1.43      1.43
2024-01-13  0.0      1.43      1.43
2024-01-14  0.0      1.43      1.43
2024-01-15  0.0      1.43      1.43
2024-01-16  0.0      1.43      1.43
2024-01-17  0.0      1.43      1.43
2024-01-18  0.0      0.00      0.00
2024-01-19  0.0      0.00      0.00
2024-01-20  0.0      0.00      0.00
2024-01-21 11.0      0.00     11.00
2024-01-22  6.0      1.57      4.43
2024-01-23  0.0      2.43      2.43
2024-01-24  0.0      2.43      2.43
2024-01-25  0.0      2.43      2.43
2024-01-26  0.0      2.4

O modelo previu zero para toda a primeira semana porque:
Janela dos 7 dias anteriores ao período
final de Dezembro/2023
nenhuma venda registrada nesses dias
média da janela = 0
previsão = 0

Para cada dia de Janeiro/2024, a previsão foi calculada como a média aritmética das vendas dos 7 dias imediatamente anteriores à data prevista. Antes disso, foi construída uma dimensão de calendário com todas as datas do período.

O modelo foi treinado exclusivamente com dados até 31/12/2023. Para cada previsão diária, a janela de 7 dias usava apenas datas estritamente anteriores à data prevista nunca a própria data ou datas futuras.

A média móvel de 7 dias não captura padrões de demanda intermitente característica desse produto, que ficou longos períodos sem venda seguidos de picos pontuais. Quando a janela se enche de zeros, a previsão converge para zero e o modelo falha exatamente nos dias mais críticos para o estoque.